Written by Nikos Meimetis. 

In [ ]:
"""node_importance_ensemble.py

Computes per-node importance scores across the full ensemble
(5 folds × 10 ensembles = 50 models).

METHOD — scLEMBAS analogue of the bulk LEMBAS "bias × activity range"
======================================================================
In bulk LEMBAS, node importance for a stimulation was:

    importance_i = |bias_i| × |Y_stim_i - Y_ctrl_i|

where bias_i is the learned static bias parameter for node i, and
(Y_stim - Y_ctrl) is the perturbation-induced change in node activity.


Conceptually: Rather than just considering the magnitude of the bias term, 
this also considers how it integrates with the rest of the model

In BioNetSC the RNN bias at each step is:

    X_bias = X_full.T + bias_global.T + bias_cats

  • X_full     – perturbation ligand signal (input_layer output)
  • bias_global – per-sample global bias from the VAE: vae(expr) → (z_mu, z_log_var, z)
  • bias_cats   – categorical (cell-type label) embedding  ← NOT USED HERE

bias_cats are still static, representing cell type specific effects
bias_global is produced freshly
each forward pass by a VAE from gene expression.  We use z_mu (the VAE's
mean, the first return value) rather than the sampled z because:
  - it is deterministic and analogous to the static bulk bias
  - it represents the EXPECTED basal activation for each node given
    the cell's expression profile, independent of stochastic sampling
  - it is NOT derived from cell-type categorical labels (that is bias_cats)
  
bias_global and bias_cat combined will represent bot the basal actiation and the cell type specific effect

The ligand-node mask (bias_mask) is applied so input node positions are
zeroed out, consistent with the forward pass.

Final formula (averaged across the Cartesian batch of stim-steps × cells
and across both cell systems):

    - global bias only: importance_i = mean_cells( |z_mu_i| × |Y_node_stim_i - Y_node_ctrl_i| )
    - categorical bias only: |bias_cat_i| × |Y_stim_i - Y_ctrl_i|
    - combined: ??

Both quantities are extracted from pure forward passes — no gradients needed.

Per-model output  (OUTPUT_DIR/fold_<F>/ensemble_<E>/):
  node_importance_ranks.csv  ←  node_name, importance_score, rank

Aggregated output (OUTPUT_DIR/):
  aggregated_node_ranks.csv  ←  node_name, n_models,
                                  median_rank, mad_rank,
                                  mean_rank,   sem_rank
  sorted ascending by median_rank (rank 1 = most important).

Resume: if node_importance_ranks.csv already exists the model is skipped.
"""

In [2]:
import os, sys, warnings, gc
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

import logging
logging.basicConfig(level=logging.INFO, format='%(message)s')
logger    = logging.getLogger()
print2log = logger.info

In [5]:
sclembas = '/home/hmbaghda/Projects/scLEMBAS'
sys.path.insert(1, os.path.join(sclembas))

sys.path.insert(1, '../../.')
from McCauley_utils import all_data, initialize_mod_and_trainer

/orcd/pool/005/hmbaghda/miniforge3/envs/scLEMBAS_v2/lib/python3.11/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator PLSRegression from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/orcd/pool/005/hmbaghda/miniforge3/envs/scLEMBAS_v2/lib/python3.11/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator OneHotEncoder from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [3]:
seed = 888
data_path = '/home/hmbaghda/orcd/pool/scLEMBAS/analysis'
author = 'McCauley'

n_cores = 30
os.environ["OMP_NUM_THREADS"] = str(1)
os.environ["MKL_NUM_THREADS"] = str(1)
os.environ["OPENBLAS_NUM_THREADS"] = str(1)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(1)
os.environ["NUMEXPR_NUM_THREADS"] = str(1)


In [13]:
# ─── Parameters ────────────────────────────────────────────────────────────
FOLDS           = [0, 1, 2, 3, 4]
N_ENSEMBLES     = 10

CONDITION       = 'Basal^TGFB1'
CONDITION_2     = 'Club^TGFB1'
PERT_NAME       = 'TGFB1'
CELL_TYPE       = 'Basal'
CELL_TYPE_2     = 'Club'

DATA_PATH       = data_path
AUTHOR          = 'McCauley'
SEED            = 888
SEED_MULT       = 21234

N_STIM_STEPS    = 21       # stimulation grid 0→1 (matches edge-importance code)
N_CONTEXT_CELLS = 200      # max cells per cell system used in the computation

OUTPUT_DIR      = os.path.join(data_path, 'processed', 'node_importance_results')

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [7]:
(sn_ppis, tf_adata, adata, expr, source_label, target_label, weight_label,
 stimulation_label, inhibition_label, cat_col, pert_col, ctrl_pert) = all_data

cond_col    = tf_adata.obs['condition']
ctrl_cond   = f'{CELL_TYPE}^{ctrl_pert}'
ctrl_cond_2 = f'{CELL_TYPE_2}^{ctrl_pert}'

basal_ctrl_barcodes = cond_col[cond_col == ctrl_cond  ].index.tolist()
club_ctrl_barcodes  = cond_col[cond_col == ctrl_cond_2].index.tolist()
print2log(f'  {ctrl_cond}  cells : {len(basal_ctrl_barcodes)}')
print2log(f'  {ctrl_cond_2} cells : {len(club_ctrl_barcodes)}')

In [14]:
def load_model(fold: int, ensemble_idx: int):
    seed_ = (SEED + ensemble_idx + 1 +
             SEED_MULT * ensemble_idx * fold) if ensemble_idx <= 3 else SEED
    mod, _ = initialize_mod_and_trainer(
        fold=fold, adversarial_penalty=True, randomize=False, seed=seed_)
    fn_base = os.path.join(DATA_PATH, 'processed', f'{AUTHOR}_fold{fold}')
    fn_mod  = (fn_base + f'model_actual_ensemble{ensemble_idx}.pt'
               if ensemble_idx < N_ENSEMBLES - 1
               else fn_base + 'model_actual.pt')
    mod.load_state_dict(torch.load(fn_mod, map_location=mod.device))
    mod.eval()
    return mod

def _get_context_tensors(barcodes, mod, device):
    expr_t  = mod.df_to_tensor(mod.expr.loc[barcodes, :]).to(device)
    cov_idx = mod.signaling_network.covariates_to_tensor(
                  sample_ids=barcodes).to(device)
    return expr_t, cov_idx


def _balance_barcodes(barcodes_1, barcodes_2, seed):
    """Subsample so neither cell system dominates the averaged importance."""
    n_each = min(len(barcodes_1), len(barcodes_2), N_CONTEXT_CELLS)
    if n_each <= 0:
        raise ValueError(
            f'Empty ctrl barcodes (cell1={len(barcodes_1)}, '
            f'cell2={len(barcodes_2)})')
    rng = np.random.default_rng(seed)
    bc1, bc2 = list(barcodes_1), list(barcodes_2)
    if len(bc1) > n_each:
        bc1 = [bc1[i] for i in rng.choice(len(bc1), n_each, replace=False)]
    if len(bc2) > n_each:
        bc2 = [bc2[i] for i in rng.choice(len(bc2), n_each, replace=False)]
    return bc1, bc2

In [68]:
# ─── Node importance: bias_mu × induced_delta ──────────────────────────────
from typing import Literal

def compute_node_importance(
    mod, pert_col_idx,
    bias_type: Literal['global', 'categorical', 'both'], # which bias terms to include in the bias part of importance_score = |bias|*delta
    ctrl_barcodes_1 = basal_ctrl_barcodes, ctrl_barcodes_2 = club_ctrl_barcodes,
    cell_type_1 = 'Basal', cell_type_2 = 'Club',
    n_stim_steps=N_STIM_STEPS,
    balance_seed=SEED
):
    """Compute |bias| × |Y_stim_i - Y_ctrl_i| for each signaling node i.

    Directly mirrors the bulk LEMBAS "bias × activity range" importance,
    translated to the BioNetSC single-cell formulation:

      bias  →  z_mu from net.vae(expr), categorical bias, or z_mu + categorical_bias depending on bias type
               (VAE mean; deterministic expression-derived basal activation;
               NOT the cell-type categorical embedding bias_cats)

      delta →  |Y_node_stim - Y_node_ctrl|
               (node-level steady-state change due to the perturbation;
               bias_cats cancels in this difference and never enters)

    Returns
    -------
    importance : np.ndarray, shape (n_nodes,)
        Mean of |bias| × |delta| across all cells and both cell systems.
    """
    mod.eval()
    device    = mod.device
    net       = mod.signaling_network
    n_ligands = len(mod.X_in.columns)

    bc1, bc2 = _balance_barcodes(ctrl_barcodes_1, ctrl_barcodes_2, balance_seed)

    stim_values = torch.linspace(0.0, 1.0, n_stim_steps + 1,
                                 device=device, dtype=mod.dtype)[1:]
    n_stim = stim_values.shape[0]

    def _build_inputs(barcodes):
        """Same Cartesian (n_stim × n_cells) batching as the edge-importance code."""
        expr_t, cov_idx = _get_context_tensors(barcodes, mod, device)
        n_cells = expr_t.shape[0]

        X_in_stim = torch.zeros(n_stim, n_ligands, device=device, dtype=mod.dtype)
        X_in_stim[:, pert_col_idx] = stim_values
        X_in_ctrl = torch.zeros(n_stim, n_ligands, device=device, dtype=mod.dtype)

        def _expand(X_in):
            return X_in.unsqueeze(1).expand(-1, n_cells, -1).reshape(-1, n_ligands)

        expr_b = expr_t.unsqueeze(0).expand(n_stim, -1, -1).reshape(-1, expr_t.shape[1])
        if cov_idx.dim() > 1:
            cov_b = cov_idx.unsqueeze(0).expand(n_stim, -1, -1).reshape(-1, cov_idx.shape[1])
        else:
            cov_b = cov_idx.unsqueeze(0).expand(n_stim, -1).reshape(-1)

        with torch.no_grad():
            X_full_stim = mod.input_layer(_expand(X_in_stim)).detach()
            X_full_ctrl = mod.input_layer(_expand(X_in_ctrl)).detach()

        return X_full_stim, X_full_ctrl, expr_b, cov_b

    X1_stim, X1_ctrl, expr1, cov1 = _build_inputs(bc1)
    X2_stim, X2_ctrl, expr2, cov2 = _build_inputs(bc2)

    with torch.no_grad():

        # ── bias_mu: expression-derived basal activation (VAE mean) ──────
        # GaussianVariationalEncoder.forward() returns (z_mu, z_log_var, z).
        # We use z_mu (the mean) rather than z (sampled) for stability and
        # determinism — analogous to the static bias in bulk LEMBAS.
        # set_seeds() inside the VAE makes z deterministic too, but z_mu is
        # the cleaner, noise-free representation of the per-cell basal state.
        #
        # The ligand-node mask zeros out input-node positions (same mask as
        # in BioNetSC.forward), ensuring consistency with the forward pass.

        bias_mu_1, _, _ = net.vae(expr1)   # (batch, n_nodes)
        bias_mu_1 = bias_mu_1.masked_fill(
            net.bias_mask.T.expand(bias_mu_1.shape[0], -1), 0.0)

        bias_mu_2, _, _ = net.vae(expr2)
        bias_mu_2 = bias_mu_2.masked_fill(
            net.bias_mask.T.expand(bias_mu_2.shape[0], -1), 0.0)

        #-----categorical bias----
        cat_biases = mod.signaling_network.cat_embeddings['cell_type']
        cat_bias_1 = cat_biases.weight[mod.signaling_network.cat_mapper['cell_type'][cell_type_1], :]
        cat_bias_2 = cat_biases.weight[mod.signaling_network.cat_mapper['cell_type'][cell_type_2], :]

        # ── node-level steady state: ctrl and stim ────────────────────────
        # net.forward() returns (Y_full, bias_tuple, t); Y_full shape (batch, n_nodes).
        # This is BEFORE the output layer — we want all signaling nodes, not
        # just the TF subset selected by the output layer.
        Y_ctrl_1 = net(X1_ctrl, cov1, expr1)[0]   # (batch, n_nodes)
        Y_stim_1 = net(X1_stim, cov1, expr1)[0]

        Y_ctrl_2 = net(X2_ctrl, cov2, expr2)[0]
        Y_stim_2 = net(X2_stim, cov2, expr2)[0]

        # ── induced activity range (delta) ────────────────────────────────
        # bias_cats is identical in stim and ctrl (same cov_idx, same expr),
        # so it cancels exactly in the difference → clean perturbation signal.
        delta_1 = (Y_stim_1 - Y_ctrl_1).abs()   # (batch, n_nodes)
        delta_2 = (Y_stim_2 - Y_ctrl_2).abs()

    #     # ── importance = |bias_mu| × |delta|, mean over cells & stim steps ─
        # TODO: scales for global bias >> delta, may need to account for that difference if using 'global' or 'both'
        if bias_type == 'global':
            imp_1 = (bias_mu_1.abs() * delta_1).mean(dim=0)   # (n_nodes,)
            imp_2 = (bias_mu_2.abs() * delta_2).mean(dim=0)
        elif bias_type == 'categorical':
            imp_1 = cat_bias_1.abs() * delta_1.mean(dim=0)
            imp_2 = cat_bias_2.abs() * delta_2.mean(dim=0)
        elif bias_type == 'both':
            imp_1 = (bias_mu_1.abs().mean(dim = 0) + cat_bias_1.abs())*delta_1.mean(dim = 0)
            imp_2 = (bias_mu_2.abs().mean(dim = 0) + cat_bias_2.abs())*delta_2.mean(dim = 0)
        else: 
            raise ValueError('Need to specify an appropriate bias_type')

        importance = ((imp_1 + imp_2) / 2).cpu().numpy()

    return importance   # (n_nodes,)

In [90]:
bias_type='categorical'

In [91]:
# ─── Main loop ─────────────────────────────────────────────────────────────
total_models = len(FOLDS) * N_ENSEMBLES
n_skipped    = 0
n_computed   = 0
n_failed     = 0

pbar = tqdm(total=total_models, desc='Node importance')

for fold in FOLDS:
    for ensemble_idx in range(N_ENSEMBLES):
        ens_dir = os.path.join(OUTPUT_DIR, f'fold_{fold}',
                               f'ensemble_{ensemble_idx}')
        out_csv = os.path.join(ens_dir, 'node_importance_ranks_{}.csv'.format(bias_type))
        pbar.set_postfix_str(f'fold={fold} ens={ensemble_idx}')

        if os.path.isfile(out_csv):
            n_skipped += 1
            pbar.update(1)
            continue

        os.makedirs(ens_dir, exist_ok=True)
        try:
            mod = load_model(fold, ensemble_idx)

            pert_cols    = list(mod.X_in.columns)
            pert_col_idx = pert_cols.index(PERT_NAME)
            node_names   = list(mod.node_idx_map.keys())

            importance = compute_node_importance(
                mod = mod, pert_col_idx = pert_col_idx,
                bias_type = bias_type, 
                ctrl_barcodes_1 = basal_ctrl_barcodes, ctrl_barcodes_2 = club_ctrl_barcodes, 
                cell_type_1 = 'Basal', cell_type_2 = 'Club',
                n_stim_steps=N_STIM_STEPS,
                balance_seed=SEED
            )

            df = pd.DataFrame({
                'node_name':        node_names,
                'importance_score': importance,
            })
            df['rank'] = df['importance_score'].rank(
                ascending=False, method='average')
            df = df.sort_values('rank').reset_index(drop=True)
            df.to_csv(out_csv, index=False)

            del mod
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()
            n_computed += 1

        except Exception as exc:
            warnings.warn(f'Failed fold={fold} ensemble={ensemble_idx}: {exc}')
            n_failed += 1

        pbar.update(1)

pbar.close()
print2log(f'\nLoop complete — computed: {n_computed}, '
          f'skipped (already on disk): {n_skipped}, failed: {n_failed}')




Node importance: 100%|████████████| 50/50 [01:54<00:00,  2.29s/it, fold=4 ens=9]


# Start

In [134]:
rows = []
for node, ranks in node_ranks.items():
    ranks   = np.asarray(ranks, dtype=float)
    n       = len(ranks)
    median_ = float(np.median(ranks))
    mad_    = float(np.median(np.abs(ranks - median_)))
    mean_   = float(np.mean(ranks))
    sem_    = (float(np.std(ranks, ddof=1) / np.sqrt(n))
               if n > 1 else float('nan'))
    rows.append({
        'node_name':   node,
        'n_models':    n,
        'median_rank': median_,
        'mad_rank':    mad_,
        'mean_rank':   mean_,
        'sem_rank':    sem_,
    })


In [135]:
# ─── Aggregation ───────────────────────────────────────────────────────────
print2log('\nAggregating ranks across models …')

node_ranks = defaultdict(list)
node_is = defaultdict(list)
n_loaded   = 0

for fold in FOLDS:
    for ensemble_idx in range(N_ENSEMBLES):
        out_csv = os.path.join(OUTPUT_DIR, f'fold_{fold}',
                               f'ensemble_{ensemble_idx}',
                               'node_importance_ranks.csv')
        if not os.path.isfile(out_csv):
            continue
        try:
            df = pd.read_csv(out_csv)
            for _, row in df.iterrows():
                node_ranks[row['node_name']].append(float(row['rank']))
                node_is[row['node_name']].append(float(row['importance_score']))
            n_loaded += 1
        except Exception as exc:
            warnings.warn(f'Could not read {out_csv}: {exc}')

print2log(f'  Loaded rank CSVs from {n_loaded} models.')

if not node_ranks:
    raise SystemExit('No per-model CSVs found — nothing to aggregate.')




In [136]:
rows = []
for node, ranks in node_ranks.items():
    ranks   = np.asarray(ranks, dtype=float)
    n       = len(ranks)
    median_ = float(np.median(ranks))
    mad_    = float(np.median(np.abs(ranks - median_)))
    mean_   = float(np.mean(ranks))
    sem_    = (float(np.std(ranks, ddof=1) / np.sqrt(n))
               if n > 1 else float('nan'))
    rows.append({
        'node_name':   node,
        'n_models':    n,
        'median_rank': median_,
        'mad_rank':    mad_,
        'mean_rank':   mean_,
        'sem_rank':    sem_,
    })

agg_df = (pd.DataFrame(rows)
            .sort_values('median_rank', ascending=True)
            .reset_index(drop=True))

out_agg = os.path.join(OUTPUT_DIR, 'aggregated_node_ranks.csv')
agg_df.to_csv(out_agg, index=False)

print2log(f'  Aggregated {len(agg_df)} nodes over {n_loaded} models.')
print2log(f'  Saved → {out_agg}')
print2log('\nTop 20 nodes by median rank:')
print2log(agg_df.head(20).to_string(index=False))

# rows = []
# for node, ranks in node_ranks.items():
#     ranks   = np.asarray(ranks, dtype=float)
#     n       = len(ranks)
#     median_ = float(np.median(ranks))
#     mad_    = float(np.median(np.abs(ranks - median_)))
#     mean_   = float(np.mean(ranks))
#     sem_    = (float(np.std(ranks, ddof=1) / np.sqrt(n))
#                if n > 1 else float('nan'))
#     rows.append({
#         'node_name':   node,
#         'n_models':    n,
#         'median_rank': median_,
#         'mad_rank':    mad_,
#         'mean_rank':   mean_,
#         'sem_rank':    sem_,
#     })


# # rows = []
# # for node, importance_score in node_is.items():
# #     importance_score = np.asarray(importance_score, dtype=float)
# #     n       = len(importance_score)
# #     median_ = float(np.median(importance_score))
# #     mad_    = float(np.median(np.abs(importance_score - median_)))
# #     mean_   = float(np.mean(importance_score))
# #     sem_    = (float(np.std(importance_score, ddof=1) / np.sqrt(n))
# #                if n > 1 else float('nan'))
# #     rows.append({
# #         'node_name': node,
# #         'n_models':  n,
# #         'median':    median_,
# #         'mad':       mad_,
# #         'mean':      mean_,
# #         'sem':       sem_,
# #     })

# # Build the dataframe FIRST
# agg_df = pd.DataFrame(rows)

# # Now add derived columns
# mu = agg_df['mean'].mean()
# sd = agg_df['mean'].std(ddof=1)
# agg_df['z_across_nodes'] = (agg_df['mean'] - mu) / sd

# higher_better = ['mean', 'median', 'mean_z', 'median_z', 'z_across_nodes']
# lower_better  = ['sem', 'mad', 'sem_z']

# for col in higher_better:
#     if col in agg_df.columns:
#         agg_df[f'{col}_rank'] = agg_df[col].rank(ascending=False, method='average')

# for col in lower_better:
#     if col in agg_df.columns:
#         agg_df[f'{col}_rank'] = agg_df[col].rank(ascending=True, method='average')

# # Now sort — `median_rank` exists because the loop above created it
# agg_df = agg_df.sort_values('median_rank', ascending=True).reset_index(drop=True)

# out_agg = os.path.join(OUTPUT_DIR, 'aggregated_node_ranks_{}.csv'.format(bias_type))
# agg_df.to_csv(out_agg, index=False)
# print2log(f'  Aggregated {len(agg_df)} nodes over {n_loaded} models.')
# print2log(f'  Saved → {out_agg}')
# print2log('\nTop 20 nodes by median rank:')
# print2log(agg_df.head(20).to_string(index=False))